# Cab Services Investment Analysis
## Which company should the investor buy?

XYZ is a private firm looking to enter the US cab market by acquiring one of two
operators: Pink Cab or Yellow Cab. This notebook works through the data they were
given and ends with a recommendation.

The data covers 359,392 completed trips between January 2016 and December 2018,
across 19 cities, spread over four files. There is no revenue or cost information
beyond what is in the trip records, so this is an analysis of unit economics and
market position, not a valuation.

Order of work:

1. Check the four files join properly and the figures are usable
2. Compare the two companies on volume, revenue and profit
3. Look at where and when each one makes money
4. Check whether rider demographics separate them
5. Test the headline differences formally
6. Model profit per trip to isolate the company effect from everything else
7. Recommend

In [ ]:
import sys
from pathlib import Path

# Lets the notebook run from notebooks/ or from the project root.
if Path.cwd().name == "notebooks":
    sys.path.insert(0, str(Path.cwd().parent))
    DATA_DIR = "../data/raw"
else:
    sys.path.insert(0, str(Path.cwd()))
    DATA_DIR = "data/raw"

import pandas as pd

from src import analysis as an
from src import model as md
from src import plots
from src.data import audit_join, build_master

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.options.display.float_format = "{:,.3f}".format

plots.setup_style()

## 1. Do the files actually join?

Four files: trips, transactions, customers, cities. Before any of the analysis is
worth reading, the joins need checking, because a bad merge silently changes every
number that follows.

In [ ]:
audit = audit_join(DATA_DIR)
for key, value in audit.items():
    print(f"{key:35} {value}")

trip_rows                           359392
transaction_rows                    440098
customer_rows                       49171
city_rows                           20
trips_without_transaction           0
transactions_without_trip           80706
duplicate_trip_ids                  0
customers_missing_demographics      0
cities_in_trips                     19
cities_in_city_file                 20
cities_with_no_trips                ['SAN FRANCISCO CA']
date_min                            2016-01-02 00:00:00
date_max                            2018-12-31 00:00:00
loss_making_trips                   24819
missing_values                      0


Four things to carry forward.

**Transaction_ID has 80,706 rows with no matching trip.** Those customer IDs exist
and the payment mode is filled in, but there is no trip record for them. The brief
gives no explanation, so the safe reading is that they belong to operators outside
the two being compared, or to trips that were cancelled. Either way they cannot
contribute to a profit comparison, so the master table is an inner join on the trip
file and those rows are dropped on purpose.

**San Francisco appears in City.csv but has no trips.** Neither company operates
there in this data, so it drops out of every city breakdown.

**24,819 trips lost money.** That is 6.9% of all trips. They are real records, not
errors, so they stay in. Which company absorbs most of them turns out to matter.

**No missing values, no duplicate trip IDs, every customer resolves.** The data is
clean past those four points.

In [ ]:
master = build_master(DATA_DIR)
print(f"{len(master):,} trips | {master['customer_id'].nunique():,} customers | "
      f"{master['city'].nunique()} cities")
master.head(3)

359,392 trips | 46,148 customers | 19 cities


,transaction_id,company,city,km_travelled,price_charged,cost_of_trip,date,profit,margin,price_per_km,cost_per_km,year,month,year_month,weekday,is_weekend,customer_id,payment_mode,gender,age,monthly_income,population,city_users,adoption_rate
0,10000011,Pink Cab,ATLANTA GA,30.450,370.950,313.635,2016-01-08,57.315,0.155,12.182,10.300,2016,1,2016-01-01,Friday,False,29290,Card,Male,28,10813,814885,24701,0.030
1,10000012,Pink Cab,ATLANTA GA,28.620,358.520,334.854,2016-01-06,23.666,0.066,12.527,11.700,2016,1,2016-01-01,Wednesday,False,27703,Card,Male,27,9237,814885,24701,0.030
2,10000013,Pink Cab,ATLANTA GA,9.040,125.200,97.632,2016-01-02,27.568,0.220,13.850,10.800,2016,1,2016-01-01,Saturday,True,28712,Cash,Male,53,11242,814885,24701,0.030


## 2. The headline comparison

Volume, revenue and profit, side by side. These three tell a different story from
each other, which is the first real finding.

In [ ]:
summary = an.company_summary(master)
summary.T

company,Pink Cab,Yellow Cab
trips,"84,711.000","274,681.000"
customers,"32,330.000","39,896.000"
revenue,"26,328,251.330","125,853,887.190"
cost,"21,020,923.009","81,833,514.019"
profit,"5,307,328.321","44,020,373.171"
mean_km,22.560,22.570
mean_fare,310.801,458.182
profit_per_trip,62.652,160.260
median_profit_per_trip,41.500,102.000
price_per_km,13.777,20.301


In [ ]:
plots.plot_market_share(master)

PosixPath('/home/claude/cab-investment-analysis/outputs/figures/01_market_share.png')

![Market share](../outputs/figures/01_market_share.png)

Yellow Cab runs 76.4% of the trips, takes 82.7% of the revenue, and keeps 89.2% of
the profit. The gap widens at every step, which means the volume lead understates
how far ahead it is on economics.

The reason is pricing, not trip length. Both companies average 22.6 km per trip,
near enough identical. Yellow Cab charges $20.30 per km against Pink Cab's $13.78,
while its cost per km is only $2.20 higher. It is charging a premium the extra cost
does not account for.

In [ ]:
plots.plot_profit_distribution(master)

PosixPath('/home/claude/cab-investment-analysis/outputs/figures/02_profit_distribution.png')

![Profit distribution](../outputs/figures/02_profit_distribution.png)

The left tail is the part worth dwelling on. 13.1% of Pink Cab trips lose money
against 5.0% of Yellow Cab's. Pink Cab is running roughly one trip in eight at a
loss, three years running.

## 3. Trend and seasonality

Levels matter less than direction when you are buying a business. Both companies
grew into 2017 and then went backwards in 2018.

In [ ]:
yearly = an.yearly_summary(master)
yearly

,company,year,trips,revenue,profit,customers,profit_per_trip,margin,profit_growth,trip_growth
0,Pink Cab,2016,25080,"7,908,479.230","1,713,511.224",16661,68.322,0.217,NaN,NaN
1,Pink Cab,2017,30321,"9,578,629.540","2,033,654.908",18643,67.071,0.212,0.187,0.209
2,Pink Cab,2018,29310,"8,841,142.560","1,560,162.189",18400,53.230,0.176,-0.233,-0.033
3,Yellow Cab,2016,82239,"38,481,133.180","13,926,995.432",25937,169.348,0.362,NaN,NaN
4,Yellow Cab,2017,98189,"45,818,910.040","16,575,977.968",27789,168.817,0.362,0.190,0.194
5,Yellow Cab,2018,94253,"41,553,843.970","13,517,399.771",27470,143.416,0.325,-0.185,-0.040


In [ ]:
plots.plot_monthly_profit(master)
plots.plot_seasonality(master)

PosixPath('/home/claude/cab-investment-analysis/outputs/figures/04_seasonality.png')

![Monthly trend](../outputs/figures/03_monthly_trend.png)

![Seasonality](../outputs/figures/04_seasonality.png)

2018 is the year that should give any buyer pause. Pink Cab's profit fell 23.3% and
Yellow Cab's fell 18.5%, on trip volumes that only dipped 3.3% and 4.0%. The volume
held up and the margin did not, which points to price pressure rather than lost
riders. Pink Cab's margin dropped from 21.2% to 17.6% and Yellow Cab's from 36.2%
to 32.5%.

Seasonality is strong and identical in shape for both. December runs 55 to 64% above
an average month, February around 35 to 50% below. Three years is thin for a seasonal
estimate, so treat the index as indicative, but the pattern is consistent enough
across all three years to plan working capital around.

## 4. Where the money is

Nineteen cities, and they are nothing like equal.

In [ ]:
city = an.city_summary(master)
pivot = an.city_pivot(master, "profit")
pivot["total"] = pivot.sum(axis=1)
pivot.sort_values("total", ascending=False).style.format("${:,.0f}")

company,Pink Cab,Yellow Cab,total
city,,,
NEW YORK NY,"$1,511,474","$26,451,081","$27,962,555"
LOS ANGELES CA,"$1,125,732","$3,285,977","$4,411,709"
WASHINGTON DC,"$193,766","$3,299,104","$3,492,870"
CHICAGO IL,"$318,722","$3,068,591","$3,387,313"
BOSTON MA,"$262,002","$1,506,718","$1,768,719"
SAN DIEGO CA,"$637,890","$949,274","$1,587,163"
SILICON VALLEY,"$340,251","$976,455","$1,316,705"
DALLAS TX,"$24,730","$1,104,003","$1,128,733"
ATLANTA GA,"$96,050","$746,383","$842,433"


In [ ]:
plots.plot_city_profit(master)
plots.plot_margin_by_city(master)
plots.plot_city_share(master)

PosixPath('/home/claude/cab-investment-analysis/outputs/figures/06_pink_city_position.png')

![City profit](../outputs/figures/05_city_profit.png)

![Margin by city](../outputs/figures/07_margin_by_city.png)

![Pink Cab city position](../outputs/figures/06_pink_city_position.png)

New York is the whole story for Yellow Cab. It produces $26.45m of profit there,
60.1% of its three-year total, at a 50.9% margin, against 40.8% in Silicon Valley, the next best.
That is the single biggest risk in the Yellow Cab case: one city regulator, one
competitor, or one licensing change could take out more than half the earnings.

Pink Cab leads on trip share in only four of nineteen cities: Nashville (61.2%),
Sacramento (56.4%), San Diego (52.1%) and Pittsburgh (51.9%). Three of those four
are small markets. Across all of them put together, Pink Cab earns less profit than
Yellow Cab earns in Dallas alone.

Tucson is the only city where Pink Cab's profit per trip beats Yellow Cab's, $74.20
against $71.53, on 799 trips. That is too small a base to build a case on.

## 5. Do the riders differ?

A common expectation going in is that the two companies serve different kinds of
customers. They do not.

In [ ]:
demographics = an.demographic_summary(master)
for dim, table in demographics.items():
    print(f"--- {dim} ---")
    print(table.to_string(index=False))
    print()

--- gender ---
   company gender  trips         profit  profit_per_trip  mean_fare  share_of_company_trips
  Pink Cab Female  37480  2,330,532.691           62.181    310.625                   0.442
  Pink Cab   Male  47231  2,976,795.630           63.026    310.941                   0.558
Yellow Cab Female 116000 18,131,417.653          156.305    454.572                   0.422
Yellow Cab   Male 158681 25,888,955.518          163.151    460.821                   0.578

--- age_band ---
   company age_band  trips         profit  profit_per_trip  mean_fare  share_of_company_trips
  Pink Cab    18-24  19306  1,227,715.516           63.592    312.103                   0.228
  Pink Cab    25-34  27072  1,700,367.567           62.809    311.870                   0.320
  Pink Cab    35-44  19270  1,192,483.565           61.883    309.327                   0.227
  Pink Cab    45-54   9169    570,172.018           62.185    310.267                   0.108
  Pink Cab    55-65   9894    616,589

In [ ]:
plots.plot_demographics(master)

PosixPath('/home/claude/cab-investment-analysis/outputs/figures/09_demographics.png')

![Demographics](../outputs/figures/09_demographics.png)

Age bands, income quartiles, gender split and payment mode are all within a couple
of percentage points of each other. Income in particular is uniform across quartiles
for both companies, which is a hint the customer attributes may have been generated
rather than observed. Whatever the cause, no rider attribute in this data separates
the two businesses, so segmentation is not where the answer is.

Loyalty does separate them.

In [ ]:
an.retention(master)

,customers,repeat_customers,mean_trips_per_customer,max_trips_per_customer,repeat_rate,share_of_trips_from_repeat_customers
company,,,,,,
Pink Cab,32330,17996,2.620,18,0.557,0.831
Yellow Cab,39896,27865,6.885,47,0.698,0.956


In [ ]:
plots.plot_retention(master)
print(an.customer_overlap(master))

{'pink_only': 6252, 'yellow_only': 13818, 'both': 26078, 'share_of_pink_customers_also_yellow': 0.8066192390968141, 'share_of_yellow_customers_also_pink': 0.6536494886705434}


![Retention](../outputs/figures/08_retention.png)

Yellow Cab customers take 6.88 trips each on average against Pink Cab's 2.62.
69.8% of Yellow Cab's customers come back for a second trip against 55.7% of Pink
Cab's, and 95.6% of Yellow Cab's volume comes from repeat riders against 83.1% for
Pink Cab.

The overlap figure is the uncomfortable one for Pink Cab: 80.7% of its customers
also rode with Yellow Cab, while only 65.4% of Yellow Cab's rode with Pink. Pink
Cab is largely serving people who use Yellow Cab too, which means it is the second
choice in its own customer base.

## 6. Testing the differences

The gaps above are large, but large samples make small gaps look significant, so
each claim gets a formal test and an effect size next to it.

In [ ]:
test = an.test_profit_per_trip(master)
for key, value in test.items():
    print(f"{key:20} {value}")

pink_mean            62.65217410961977
yellow_mean          160.25998584103016
difference           97.60781173141038
welch_t              230.99551452746311
welch_p              0.0
mannwhitney_u        16084098181.0
mannwhitney_p        0.0
cohens_d             0.6302708587983085
n_pink               84711
n_yellow             274681


In [ ]:
for key, value in an.test_margin(master).items():
    print(f"{key:20} {value}")

pink_mean_margin     0.17380832562291507
yellow_mean_margin   0.30134544234470556
difference           0.1275371167217905
welch_t              199.14466870385888
welch_p              0.0


In [ ]:
for column in ["gender", "payment_mode"]:
    result = an.test_independence(master, column)
    print(f"--- {column} ---")
    print(result["table"])
    print(f"chi2 = {result['chi2']:.2f}   p = {result['p']:.4f}   "
          f"Cramer's V = {result['cramers_v']:.4f}\n")

--- gender ---


gender      Female    Male
company                   
Pink Cab     37480   47231
Yellow Cab  116000  158681
chi2 = 107.22   p = 0.0000   Cramer's V = 0.0173

--- payment_mode ---
payment_mode    Card    Cash
company                     
Pink Cab       50719   33992
Yellow Cab    164785  109896
chi2 = 0.37   p = 0.5412   Cramer's V = 0.0010



In [ ]:
an.test_city_profit_differences(master)

{'Pink Cab': {'f': 531.3167193452098, 'p': 0.0, 'cities': 19},
 'Yellow Cab': {'f': 8780.983415520906, 'p': 0.0, 'cities': 19}}

What the tests say.

**Profit per trip.** Yellow Cab earns $97.61 more per trip. Welch's t-test and
Mann-Whitney both return p below 0.001, and Cohen's d is 0.63, a moderate effect.
With 359,392 rows the p-value was never going to be interesting; the effect size is
what makes the gap worth acting on.

**Margin.** Yellow Cab's mean trip-level margin is 12.8 percentage points higher,
p below 0.001. Note this is the average of per-trip margins, which weights a $20
trip the same as a $500 one. The aggregate margin, total profit over total revenue,
is 35.0% against 20.2%. Both measures point the same way.

**Gender.** The chi-square test returns p below 0.0001, but Cramer's V is 0.017.
That is a statistically detectable association with no practical content, and it is
a clean example of why significance alone should not drive a decision at this sample
size. Treat gender as unrelated to company choice.

**Payment mode.** p = 0.54. No association at all. Card and cash split roughly 60/40
for both.

**City.** One-way ANOVA on profit per trip across cities is significant for both
companies, which confirms that city is a real driver and that any company comparison
has to control for it. That is what the model does next.

## 7. Modelling profit per trip

The descriptive work shows Yellow Cab earns more per trip. It does not show how much
of that is the company and how much is the mix of cities and trip lengths it happens
to serve. A model that holds those constant separates the two.

Two rules.

Profit is price minus cost, so neither of those columns, nor margin or price per km,
can be a feature. Only what is knowable before the meter runs goes in.

The split is by time, not at random. 2016 and 2017 train, 2018 is held out. A random
split would leak the 2018 margin environment into training and flatter the result,
and 2018 is exactly the year a buyer cares about.

In [ ]:
results, fitted, (X_train, y_train, X_test, y_test) = md.evaluate(master)
print(f"train {len(X_train):,} trips   test {len(X_test):,} trips")
results

train 235,829 trips   test 123,563 trips


,train_mae,test_mae,train_rmse,test_rmse,train_r2,test_r2,test_bias
model,,,,,,,
Mean baseline,123.143,115.137,167.375,146.517,0.000,-0.026,23.210
Distance only (linear),103.013,98.797,147.804,131.471,0.220,0.174,23.037
"Ridge, all features",77.887,68.670,106.101,90.269,0.598,0.611,21.200
Gradient boosting,52.146,42.825,75.832,62.367,0.795,0.814,20.916


In [ ]:
plots.plot_model_performance(results)

PosixPath('/home/claude/cab-investment-analysis/outputs/figures/10_model_performance.png')

![Model performance](../outputs/figures/10_model_performance.png)

Gradient boosting lands at $42.82 mean absolute error and R-squared of 0.814 on the
held-out year, against $98.80 and 0.174 for a distance-only baseline. The complexity
earns its place.

The column worth reading twice is `test_bias`. Every model overpredicts 2018 profit,
the best one by $20.92 per trip. That is not a bug, it is the margin squeeze showing
up: a model trained on 2016 and 2017 expects trips to be more profitable than they
actually were in 2018. Anyone forecasting forward from these three years should
assume the 2018 level, not the three-year average.

In [ ]:
importance = md.feature_importance(fitted["Gradient boosting"], X_test, y_test)
importance

,feature,mae_increase,std
0,km_travelled,54.404,0.239
1,population,42.574,0.216
2,company,18.813,0.221
3,month,12.595,0.214
4,city,3.246,0.093
5,adoption_rate,2.946,0.034
6,weekday,2.879,0.031
7,age,0.015,0.006
8,gender,0.006,0.004
9,payment_mode,-0.000,0.004


In [ ]:
plots.plot_feature_importance(importance)
plots.plot_predicted_vs_actual(fitted["Gradient boosting"], X_test, y_test)

PosixPath('/home/claude/cab-investment-analysis/outputs/figures/12_predicted_vs_actual.png')

![Feature importance](../outputs/figures/11_feature_importance.png)

![Predicted vs actual](../outputs/figures/12_predicted_vs_actual.png)

Distance dominates, which is what you would expect. City population comes second and
city name fifth, which looks odd until you notice they encode the same thing: the
tree can split a continuous population variable more finely than it can use nineteen
one-hot city flags, so population absorbs most of the city signal and the city column
is left with the residual. Read the two together as one city effect of roughly $46.

Company ranks third at $18.81. Rider attributes contribute nothing: age, gender,
income and payment mode all sit at or below zero, which matches what the descriptive
work found.

The last question is what the company badge is worth on its own.

In [ ]:
# Score the held-out trips as they are, then score the same trips with the
# company swapped. The difference is the company effect with city, distance,
# date and rider held constant.
md.company_effect(fitted["Gradient boosting"], X_test)

,mean,median,count
company,,,
Pink Cab,76.691,47.275,29310
Yellow Cab,96.754,52.632,94253


Holding everything else constant, a trip run by Yellow Cab is worth $47 to $53 more
at the median than the same trip run by Pink Cab, and $77 to $97 more on the mean.
The mean sits well above the median because the New York trips pull it up.

So the raw $97.61 gap is partly mix. Correct for city, distance, timing and rider,
and roughly half of it remains and is attributable to the company itself: its
pricing power, its position, or whatever is behind them.

## 8. Recommendation

**Acquire Yellow Cab.**

The case rests on four things the data supports directly.

Yellow Cab earns $44.0m of profit against Pink Cab's $5.3m over the three years, at
a 35.0% aggregate margin against 20.2%. It out-earns Pink Cab in all 19 cities.
It keeps its riders: 69.8% come back, against 55.7%, and its customers take 2.6
times as many trips each. And when city, distance, timing and rider are held
constant, the company effect is still worth about $50 per trip at the median, so the
advantage belongs to the company and not to the cities it happens to serve.

Pink Cab is not a cheaper route to the same place. It runs 13.1% of its trips at a
loss, it leads in four small cities, and 80.7% of its customers already ride with
Yellow Cab, which makes it the second choice inside its own base.

Three conditions should go with the recommendation.

**Price New York properly.** 60.1% of Yellow Cab's profit comes from one city at a
margin no other city approaches. That concentration is the main risk in the deal and
should be stress-tested in the valuation, not averaged away.

**Use 2018 as the baseline, not the three-year average.** Both companies lost
roughly a fifth of their profit in 2018 while volumes barely moved. The model
confirms it independently by overpredicting the held-out year by $20.92 per trip.
Underwriting on 2016 and 2017 economics would overstate the business.

**Do not expect segmentation gains.** Age, income, gender and payment mode are flat
across both companies. The levers in this business are pricing, city mix and
retention.

### What this analysis cannot tell you

Fleet size, driver counts, fixed costs, licensing and capital structure are all
absent, so "profit" here is trip contribution, not company profit. There is no
competitor data, so the 2018 margin compression has no identified cause. The rider
attributes look generated rather than observed, which is why no conclusion here
rests on them. And three years is short for a seasonal estimate. A confirmatory
diligence pass would need the cost base and at least two more years.